In [1]:
import numpy as np

In [2]:
import apple

In [3]:
corpus = """the king rules the realm
the queen rules the realm
the king is a man
the queen is a woman
the man drinks beer
the woman drinks wine
the king drinks wine
the queen drinks wine
a man walks the road
a woman walks the road"""

In [4]:
sentences = [s.split() for s in corpus.splitlines()]
vocab = sorted({w for s in sentences for w in s})
ix = {w: i for i, w in enumerate(vocab)}
V = len(vocab)
V

14

In [5]:
# skip-gram pairs, window 2
centers, contexts = [], []
for s in sentences:
    for i, w in enumerate(s):
        for j in range(max(0, i - 2), min(len(s), i + 3)):
            if j != i:
                centers.append(ix[w]); contexts.append(ix[s[j]])
centers = np.array(centers, dtype=np.int64)
contexts = np.array(contexts, dtype=np.int64)
len(centers)

124

In [6]:
disjunctify = apple.jit("λxs.λN.(λn.[?x=n,.1::float,.0]'⍳N)'(xs::Vec n int)")
X = disjunctify(centers,V-1)
C = disjunctify(contexts,V-1)

In [7]:
step = apple.jit('''
λX.λC.λw1.λw2.λη.
{
  h ⟜ X%.w1;
  u ← h%.w2;
  softmax ← λv. {m ⟜ (⋉)/v; es ⟜ [e:(x-m)]'v; s ⟜ (+)/es; (%s)'es};
  out ← softmax'u;
  n ⟜ ℝ(𝓉 X);
  e ⟜ [(x-y)%n]`{0,0} out C;
  δw2 ← (⍉h)%.e;
  δw1 ← (⍉X)%.(e%.(⍉w2));
  ([x-η*y]`{0,0} w1 δw1, [x-η*y]`{0,0} w2 δw2)
}
''')

In [8]:
loss = apple.jit('''
λX.λC.λw1.λw2.
{
  u ← (X%.w1)%.w2;
  softmax ← λv. {m ⟜ (⋉)/v; es ⟜ [e:(x-m)]'v; s ⟜ (+)/es; (%s)'es};
  out ← softmax'u;
  _((+)/* 0 ((*)`{0,0} C (_.`{0} out)))%ℝ(𝓉 X)
}
''')

In [9]:
D = 8
rng = np.random.default_rng(17)
w1 = rng.uniform(-1,1,(V,D))/np.sqrt(V*D)
w2 = rng.uniform(-1,1,(D,V))/np.sqrt(D*V)

In [10]:
# (w1,w2)=apple.jit("{φ ← 1%√(8*14); ((*φ)`{0} (𝔯 _1 1 :: Arr (14×8) float), (*φ)`{0} (𝔯 _1 1 :: Arr (8×14) float))}")()

In [11]:
for epoch in range(1001):
    if epoch % 200 == 0:
        print(f"epoch {epoch:4d}  loss {loss(X, C, w1, w2):.4f}")
    w1, w2 = step(X, C, w1, w2, 1.0)

epoch    0  loss 2.6398
epoch  200  loss 1.7945
epoch  400  loss 1.7024
epoch  600  loss 1.6868
epoch  800  loss 1.6817
epoch 1000  loss 1.6799


In [12]:
norm=apple.jit("([{φ ⟜ √(x⋅x);(%φ)'x}]')")

In [13]:
def neighbours(w, k=3):
    e = norm(w1)
    sims = e @ e[ix[w]]
    return [vocab[i] for i in np.argsort(-sims)[1:k+1]]

for w in ("king", "man", "beer"):
    print(f"{w:>6} ~ {neighbours(w)}")

  king ~ ['queen', 'woman', 'realm']
   man ~ ['woman', 'road', 'king']
  beer ~ ['wine', 'the', 'a']
